# AI Resume Screening System with LangSmith Tracing
**Innomatics Research Labs — Data Science Internship (Feb 2026)**

**Stack**: Groq (LLaMA 3) · LangChain LCEL · LangSmith

**Pipeline**: Resume → Skill Extraction → Matching → Scoring → Explanation → Tracing

## 1. Install Dependencies

In [17]:
# Run once to install required packages
!pip install langchain langchain-groq langsmith python-dotenv -q

## 2. Environment Setup

In [18]:
import os
from getpass import getpass

# ─── Fill in your keys here (or use a .env file) ───────────────────
os.environ["GROQ_API_KEY"]         = getpass("Enter Groq API Key: ")
os.environ["LANGCHAIN_API_KEY"]    = getpass("Enter LangSmith API Key: ")
os.environ["LANGCHAIN_TRACING_V2"]    = "true"
os.environ["LANGCHAIN_PROJECT"]       = "resume-screener"
os.environ["LANGCHAIN_ENDPOINT"]      = "https://api.smith.langchain.com"
# ───────────────────────────────────────────────────────────────────

print("Environment configured ✓")
print(f"LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"Tracing Enabled:   {os.environ['LANGCHAIN_TRACING_V2']}")

Enter Groq API Key:  ········
Enter LangSmith API Key:  ········


Environment configured ✓
LangSmith Project: resume-screener
Tracing Enabled:   true


## 3. Job Description & Sample Resumes

In [19]:
JOB_DESCRIPTION = """
Position: Data Scientist | Company: TechCorp Analytics

Required Skills:
- Python (3+ years), SQL
- Machine Learning: scikit-learn, XGBoost
- Deep Learning: TensorFlow or PyTorch
- Data visualization: Matplotlib, Seaborn, or Tableau
- Feature engineering, model evaluation, Statistics, Pandas, NumPy

Nice-to-Have: Apache Spark, Cloud (AWS/GCP/Azure), MLflow, NLP
Experience: 2+ years in data science or ML
Education: Bachelor's or Master's in CS, Statistics, Math, or related
"""

RESUME_STRONG = """
Name: Arjun Mehta
SKILLS: Python, SQL, R, scikit-learn, XGBoost, TensorFlow, PyTorch, Pandas, NumPy,
        Matplotlib, Seaborn, Tableau, Apache Spark, AWS SageMaker, GCP BigQuery,
        MLflow, Docker, HuggingFace Transformers, spaCy
EXPERIENCE: Data Scientist at DataDriven Inc. (3 years) — deployed 10+ production ML
models, demand forecasting with LSTM, Spark feature engineering, MLflow tracking.
Junior Data Analyst (1 year) — SQL analysis, Tableau dashboards.
EDUCATION: M.Sc. Data Science — IIT Bombay (2020)
CERTIFICATIONS: AWS ML Specialty, Google Professional Data Engineer
PROJECTS: Fraud Detection (XGBoost+Kafka), Sentiment Analysis API (BERT), Churn Prediction
"""

RESUME_AVERAGE = """
Name: Priya Sharma
SKILLS: Python, SQL, Pandas, NumPy, Matplotlib, Seaborn, scikit-learn, MySQL, Power BI
EXPERIENCE: Data Analyst at InfoSoft Solutions (2 years) — EDA, K-Means clustering,
SQL queries, Power BI reports. Intern at StartupX (6 months) — data cleaning, regression.
EDUCATION: B.Tech Information Technology — Anna University (2021)
PROJECTS: House Price Prediction (Linear Regression), Sales Dashboard (SQL + Power BI)
"""

RESUME_WEAK = """
Name: Rahul Verma
SKILLS: Python (basic), HTML, CSS, Microsoft Excel, Google Sheets
EXPERIENCE: No professional experience in data science or ML.
EDUCATION: B.Com — Delhi University (2023)
ONLINE COURSES: Python for Beginners (Udemy), Excel for Data Analysis (Coursera)
PROJECTS: Student Grade Calculator (Python script), Monthly Expense Tracker (Excel)
"""

print("Data loaded ✓")

Data loaded ✓


## 4. Prompt Templates (PromptTemplate + LCEL)

In [20]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import json, re

# Force JSON mode at the LLM level — most reliable fix
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    model_kwargs={"response_format": {"type": "json_object"}}
)

def safe_json(raw):
    cleaned = re.sub(r'```(?:json)?', '', raw).strip().rstrip('`').strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start == -1 or end == -1:
        raise ValueError(f"No JSON found:\n{cleaned}")
    return json.loads(cleaned[start:end+1])
    
# ── Step 1: Skill Extraction ────────────────────────────────────────
extraction_prompt = PromptTemplate(
    input_variables=["resume_text"],
    template="""You are a precise resume parser. Extract ONLY what is EXPLICITLY in the resume.
RULES: Do NOT infer skills. Output ONLY valid JSON.
Resume: {resume_text}
Return: {{"candidate_name": "", "skills": [], "tools_and_technologies": [],
          "experience_years": "", "education": "", "certifications": [], "projects": []}}"""
)
extraction_chain = extraction_prompt | llm | StrOutputParser()

# ── Step 2: Matching ────────────────────────────────────────────────
matching_prompt = PromptTemplate(
    input_variables=["job_description", "candidate_profile"],
    template="""You are a technical recruiter doing a skills-gap analysis.
Job Description: {job_description}
Candidate Profile: {candidate_profile}
RULES: Only use data from candidate profile. Output ONLY valid JSON.
Return: {{"matched_skills": [], "missing_skills": [], "bonus_skills": [],
          "experience_match": "", "education_match": ""}}"""
)
matching_chain = matching_prompt | llm | StrOutputParser()

# ── Step 3: Scoring ─────────────────────────────────────────────────
scoring_prompt = PromptTemplate(
    input_variables=["matching_result", "candidate_profile", "job_description"],
    template="""You are a hiring evaluator. Score using: Skills Match=40, Experience=25,
Education=15, Bonus Skills=10, Projects=10. Total=100.
Matching: {matching_result} | Profile: {candidate_profile} | JD: {job_description}
Output ONLY valid JSON.
Return: {{"total_score": 0, "score_breakdown": {{"skills_match_score": 0,
  "experience_score": 0, "education_score": 0, "bonus_skills_score": 0,
  "projects_score": 0}}, "rating": ""}}"""
)
scoring_chain = scoring_prompt | llm | StrOutputParser()

# ── Step 4: Explanation (with few-shot) ─────────────────────────────
explanation_prompt = PromptTemplate(
    input_variables=["candidate_profile", "matching_result", "score_result"],
    template="""You are a senior recruiter writing a hiring recommendation.
EXAMPLE — Score 82/Strong: Candidate is a strong fit with solid ML skills coverage...
Recommend: Proceed to technical interview.

Now evaluate: Profile={candidate_profile} Matching={matching_result} Score={score_result}
RULES: Be factual. Reference only data above. Output ONLY valid JSON.
Return: {{"summary": "", "strengths": [], "weaknesses": [],
          "recommendation": "", "recommendation_reason": ""}}"""
)
explanation_chain = explanation_prompt | llm | StrOutputParser()

print("All LCEL chains built ✓")

All LCEL chains built ✓


## 5. Pipeline Runner

In [21]:
def run_pipeline(resume_text, job_description, label):
    """Run the 4-step pipeline for one resume."""
    tags = [label.lower(), 'resume-screener', 'innomatics']
    cfg = {'tags': tags}
    
    print(f"\n{'='*55}")
    print(f"  Candidate: {label}")
    print(f"{'='*55}")
    
    # Step 1
    print("[1/4] Extracting skills...")
    extracted = safe_json(extraction_chain.invoke({'resume_text': resume_text}, config=cfg))
    
    # Step 2
    print("[2/4] Matching skills...")
    matched = safe_json(matching_chain.invoke({
        'job_description': job_description,
        'candidate_profile': json.dumps(extracted)
    }, config=cfg))
    
    # Step 3
    print("[3/4] Scoring...")
    scored = safe_json(scoring_chain.invoke({
        'matching_result': json.dumps(matched),
        'candidate_profile': json.dumps(extracted),
        'job_description': job_description
    }, config=cfg))
    
    # Step 4
    print("[4/4] Generating explanation...")
    explained = safe_json(explanation_chain.invoke({
        'candidate_profile': json.dumps(extracted),
        'matching_result': json.dumps(matched),
        'score_result': json.dumps(scored)
    }, config=cfg))
    
    # Print result
    print(f"\n  Score: {scored['total_score']}/100 ({scored['rating']})")
    print(f"  Matched: {matched['matched_skills']}")
    print(f"  Missing: {matched['missing_skills']}")
    print(f"  Recommendation: {explained['recommendation']}")
    print(f"  Summary: {explained['summary']}")
    
    return {'label': label, 'extracted': extracted, 'matched': matched,
            'scored': scored, 'explained': explained}

print('Runner ready ✓')

Runner ready ✓


## 6. Run All Three Candidates (LangSmith traces 3 runs)

In [22]:
results = []
for label, resume in [('Strong', RESUME_STRONG), ('Average', RESUME_AVERAGE), ('Weak', RESUME_WEAK)]:
    r = run_pipeline(resume, JOB_DESCRIPTION, label)
    results.append(r)


  Candidate: Strong
[1/4] Extracting skills...
[2/4] Matching skills...
[3/4] Scoring...
[4/4] Generating explanation...

  Score: 90/100 (Highly Qualified)
  Matched: ['Python', 'SQL', 'scikit-learn', 'XGBoost', 'TensorFlow', 'PyTorch', 'Pandas', 'NumPy', 'Matplotlib', 'Seaborn', 'Tableau']
  Missing: ['Apache Spark', 'Cloud (AWS/GCP/Azure)', 'MLflow', 'NLP']
  Recommendation: Proceed to technical interview.
  Summary: Candidate Arjun Mehta is a highly qualified fit for the role.

  Candidate: Average
[1/4] Extracting skills...
[2/4] Matching skills...
[3/4] Scoring...
[4/4] Generating explanation...

  Score: 70/100 (Good)
  Matched: ['Python', 'SQL', 'Pandas', 'NumPy', 'Matplotlib', 'Seaborn', 'scikit-learn']
  Missing: ['XGBoost', 'TensorFlow or PyTorch', 'Feature engineering', 'model evaluation', 'Statistics']
  Recommendation: Proceed to technical interview with caution.
  Summary: Candidate Priya Sharma has a good foundation in ML skills, but lacks some advanced skills.

  Cand

## 7. Summary Table

In [24]:
print(f"\n{'='*65}")
print(f"  {'Candidate':<10} {'Score':>7}   {'Rating':<10} {'Recommendation'}")
print(f"  {'-'*63}")
for r in results:
    name = r['extracted'].get('candidate_name', r['label'])
    print(f"  {name:<20} {r['scored']['total_score']:>5}/100  "
          f"{r['scored']['rating']:<10} {r['explained']['recommendation']}")


  Candidate    Score   Rating     Recommendation
  ---------------------------------------------------------------
  Arjun Mehta             90/100  Highly Qualified Proceed to technical interview.
  Priya Sharma            70/100  Good       Proceed to technical interview with caution.
  Rahul Verma              0/100  Not Qualified Not Qualified


## 8. LangSmith Trace Verification

After running the cells above, go to [https://smith.langchain.com](https://smith.langchain.com)
and open project **resume-screener**. You should see:
- ✅ 3 parent runs (Strong, Average, Weak)
- ✅ 4 child spans per run (extraction, matching, scoring, explanation)
- ✅ Tags: `strong`, `average`, `weak`, `resume-screener`, `innomatics`

## 9. Debug: Incorrect Output Demo

The cell below intentionally feeds a mismatched resume to demonstrate
LangSmith catching a case where the model scores too high (hallucination guard).

In [25]:
# Debug run — weak resume fed with wrong label to show incorrect output detection
print("[DEBUG] Running with deliberately mismatched input...")
debug_result = run_pipeline(RESUME_WEAK, JOB_DESCRIPTION, 'Debug-IncorrectExpected')
print(f"\nExpected: Weak/Reject | Got: {debug_result['scored']['rating']} / {debug_result['explained']['recommendation']}")
print("Check LangSmith for the tagged debug trace.")

[DEBUG] Running with deliberately mismatched input...

  Candidate: Debug-IncorrectExpected
[1/4] Extracting skills...
[2/4] Matching skills...
[3/4] Scoring...
[4/4] Generating explanation...

  Score: 0/100 (Not Qualified)
  Matched: ['Python']
  Missing: ['SQL', 'scikit-learn', 'XGBoost', 'TensorFlow', 'PyTorch', 'Matplotlib', 'Seaborn', 'Tableau', 'Feature engineering', 'model evaluation', 'Statistics', 'Pandas', 'NumPy']
  Recommendation: Not Qualified
  Summary: Candidate has basic Python skills and some experience with data manipulation tools like Excel and Google Sheets.

Expected: Weak/Reject | Got: Not Qualified / Not Qualified
Check LangSmith for the tagged debug trace.
